# 03: Probing Experiments — Main Results

**Goal**: Train probing classifiers on frozen embeddings and generate the core experimental results.

This notebook:
1. Loads cached embeddings from all models
2. Trains logistic regression probes for each relation type
3. Reports F1 scores per model × relation
4. Creates the main results comparison table
5. Analyzes which embedding dimensions are diagnostic per relation

All experimental design principles from CLAUDE.md:
- Stratified k-fold CV (k=5) to handle class imbalance
- Binary probing per relation (cleaner signal than multiclass)
- Report macro F1 (class-weighted) not just accuracy
- Keep regularization weak (high C) to test representation, not regularize it away

## 1. Setup and Load Cached Embeddings

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os

# Setup
sys.path.insert(0, str(Path.cwd().parent))
load_dotenv()
CACHE_DIR = Path(os.getenv("CACHE_DIR", "../results/embeddings"))
RESULTS_DIR = Path("../results")

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)

print(f"Cache directory: {CACHE_DIR}")
print(f"Available embeddings:")
for f in sorted(CACHE_DIR.glob("*.npy")):
    size_mb = f.stat().st_size / (1024*1024)
    print(f"  {f.name:30s} {size_mb:8.1f} MB")

## 2. Load Embeddings and Dataset Labels

In [ ]:
from datasets import load_dataset

# Load embeddings
print("Loading cached embeddings...")
sbert_embeddings = np.load(CACHE_DIR / "sbert_vsr_train.npy")
clip_text_embeddings = np.load(CACHE_DIR / "clip_text_vsr_train.npy")
clip_mm_embeddings = np.load(CACHE_DIR / "clip_multimodal_vsr_train.npy")

print(f"Embedding shapes:")
print(f"  SBERT:        {sbert_embeddings.shape}")
print(f"  CLIP text:    {clip_text_embeddings.shape}")
print(f"  CLIP mm:      {clip_mm_embeddings.shape}")

# Load VSR dataset for relation types and labels
print(f"\nLoading VSR dataset...")
vsr = load_dataset("cambridgeltl/vsr_random")
train_data = vsr["train"]

# Extract labels
relation_types = np.array([ex["relation_type"] for ex in train_data])
binary_labels = np.array([int(ex["label"]) for ex in train_data])

print(f"Dataset: {len(train_data)} examples")
print(f"Relation types: {len(np.unique(relation_types))} unique")
print(f"Label dist: {np.sum(binary_labels)} True, {len(binary_labels) - np.sum(binary_labels)} False")

## 3. Train Probes for Each Model

In [ ]:
from src.probing import probe_by_relation_type, model_comparison_table

print("Training SBERT probe...")
sbert_results = probe_by_relation_type(
    sbert_embeddings,
    relation_types,
    binary_labels,
    n_splits=5,
    C=1.0,
    min_samples=20
)
print(f"  Relations tested: {len(sbert_results)}")
print(f"  Mean F1: {sbert_results['f1_macro_mean'].mean():.3f}")

print(f"\nTraining CLIP text probe...")
clip_text_results = probe_by_relation_type(
    clip_text_embeddings,
    relation_types,
    binary_labels,
    n_splits=5,
    C=1.0,
    min_samples=20
)
print(f"  Relations tested: {len(clip_text_results)}")
print(f"  Mean F1: {clip_text_results['f1_macro_mean'].mean():.3f}")

print(f"\nTraining CLIP multimodal probe...")
clip_mm_results = probe_by_relation_type(
    clip_mm_embeddings,
    relation_types,
    binary_labels,
    n_splits=5,
    C=1.0,
    min_samples=20
)
print(f"  Relations tested: {len(clip_mm_results)}")
print(f"  Mean F1: {clip_mm_results['f1_macro_mean'].mean():.3f}")

## 4. Main Results Table

In [ ]:
# Combine results across models
results_dict = {
    "sbert": sbert_results,
    "clip_text": clip_text_results,
    "clip_mm": clip_mm_results,
}

comparison = model_comparison_table(results_dict)
print(comparison.head(15).to_string())

# Save results table
comparison.to_csv(RESULTS_DIR / "probing_results.csv", index=False)
print(f"\nSaved results table to results/probing_results.csv")

## 5. Summary Statistics

In [ ]:
# Compute statistics per model
print("\n" + "="*70)
print("PROBING RESULTS SUMMARY")
print("="*70)

for model_name, results_df in results_dict.items():
    f1_scores = results_df["f1_macro_mean"].values
    print(f"\n{model_name.upper():15s}:")
    print(f"  Mean F1:       {f1_scores.mean():.3f} (±{f1_scores.std():.3f})")
    print(f"  Median F1:     {np.median(f1_scores):.3f}")
    print(f"  Min F1:        {f1_scores.min():.3f}")
    print(f"  Max F1:        {f1_scores.max():.3f}")
    print(f"  Relations:     {len(results_df)}")

# Compute CLIP improvements
print(f"\nCLIP vs SBERT Improvements:")
common_relations = set(sbert_results["relation"]) & set(clip_text_results["relation"]) & set(clip_mm_results["relation"])
print(f"  Common relations tested: {len(common_relations)}")

sbert_f1_dict = dict(zip(sbert_results["relation"], sbert_results["f1_macro_mean"]))
clip_text_f1_dict = dict(zip(clip_text_results["relation"], clip_text_results["f1_macro_mean"]))
clip_mm_f1_dict = dict(zip(clip_mm_results["relation"], clip_mm_results["f1_macro_mean"]))

text_improvements = [clip_text_f1_dict[r] - sbert_f1_dict[r] for r in common_relations]
mm_improvements = [clip_mm_f1_dict[r] - sbert_f1_dict[r] for r in common_relations]

print(f"  CLIP text vs SBERT: {np.mean(text_improvements):+.3f} (±{np.std(text_improvements):.3f})")
print(f"  CLIP mm vs SBERT:   {np.mean(mm_improvements):+.3f} (±{np.std(mm_improvements):.3f})")
print(f"  CLIP mm vs CLIP text: {(np.mean(mm_improvements) - np.mean(text_improvements)):+.3f}")

print("\n" + "="*70)

## 6. Visualize Results as Heatmap

In [ ]:
# Prepare data for heatmap
model_names = ["sbert", "clip_text", "clip_mm"]
heatmap_data = comparison[model_names + "_f1"].copy()
heatmap_data.columns = ["SBERT", "CLIP Text", "CLIP Multimodal"]

# Select top relations by mean F1
top_n = 20
top_relations = heatmap_data.mean(axis=1).nlargest(top_n).index
heatmap_data = heatmap_data.loc[top_relations]

# Create heatmap
fig, ax = plt.subplots(figsize=(8, 12))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    vmin=0,
    vmax=1,
    cbar_kws={"label": "F1 Score"},
    ax=ax,
    linewidths=0.5
)
ax.set_title(f"Probing Results: Top {top_n} Relations (by mean F1)", fontsize=14, fontweight="bold")
ax.set_xlabel("Model", fontsize=12)
ax.set_ylabel("Relation Type", fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "figures" / "03_results_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved: 03_results_heatmap.png")

## 7. Detailed Relation Analysis

In [ ]:
# Show relations where CLIP provides most benefit
comparison_copy = comparison.copy()
comparison_copy["clip_text_gain"] = comparison_copy["clip_text_f1"] - comparison_copy["sbert_f1"]
comparison_copy["clip_mm_gain"] = comparison_copy["clip_mm_f1"] - comparison_copy["sbert_f1"]

print("\nRelations with LARGEST CLIP TEXT gains:")
top_text_gains = comparison_copy.nlargest(5, "clip_text_gain")[["relation", "sbert_f1", "clip_text_f1", "clip_text_gain"]]
for idx, row in top_text_gains.iterrows():
    print(f"  {row['relation']:25s}: {row['sbert_f1']:.3f} → {row['clip_text_f1']:.3f} (+{row['clip_text_gain']:.3f})")

print(f"\nRelations with LARGEST CLIP MULTIMODAL gains:")
top_mm_gains = comparison_copy.nlargest(5, "clip_mm_gain")[["relation", "sbert_f1", "clip_mm_f1", "clip_mm_gain"]]
for idx, row in top_mm_gains.iterrows():
    print(f"  {row['relation']:25s}: {row['sbert_f1']:.3f} → {row['clip_mm_f1']:.3f} (+{row['clip_mm_gain']:.3f})")

print(f"\nRelations where SBERT outperforms CLIP:")
sbert_better = comparison_copy[comparison_copy["sbert_f1"] > comparison_copy["clip_mm_f1"]]
if len(sbert_better) > 0:
    for idx, row in sbert_better.head(5).iterrows():
        print(f"  {row['relation']:25s}: SBERT {row['sbert_f1']:.3f} vs CLIP mm {row['clip_mm_f1']:.3f}")
else:
    print("  (None - CLIP multimodal superior across all relations)")

## 8. Save Detailed Results

In [ ]:
# Save detailed results for each model
for model_name, results_df in results_dict.items():
    csv_path = RESULTS_DIR / f"probing_results_{model_name}_detailed.csv"
    results_df.to_csv(csv_path, index=False)
    print(f"Saved detailed results: {csv_path.name}")

# Save comparison with gains
comparison_copy.to_csv(RESULTS_DIR / "probing_results_with_gains.csv", index=False)
print(f"Saved comparison with gains: probing_results_with_gains.csv")

## 9. Key Findings

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)

print(f"\n1. OVERALL PERFORMANCE:")
for model_name in ["sbert", "clip_text", "clip_mm"]:
    f1 = comparison[f"{model_name}_f1"].mean()
    print(f"   {model_name.upper():20s}: {f1:.1%} mean F1")

print(f"\n2. VISUAL GROUNDING BENEFIT:")
text_gain = comparison["clip_text_f1"].mean() - comparison["sbert_f1"].mean()
mm_gain = comparison["clip_mm_f1"].mean() - comparison["sbert_f1"].mean()
print(f"   CLIP text vs SBERT:  {text_gain:+.1%}")
print(f"   CLIP mm vs SBERT:    {mm_gain:+.1%}")
print(f"   Image addition (mm vs text): {(mm_gain - text_gain):+.1%}")

print(f"\n3. RELATION-SPECIFIC INSIGHTS:")
hardest_sbert = comparison.nsmallest(3, "sbert_f1")[["relation", "sbert_f1"]]
print(f"   Hardest for SBERT (lowest F1):")
for idx, row in hardest_sbert.iterrows():
    print(f"     - {row['relation']:25s}: {row['sbert_f1']:.3f}")

easiest = comparison.nlargest(3, "clip_mm_f1")[["relation", "clip_mm_f1"]]
print(f"   Easiest for CLIP mm (highest F1):")
for idx, row in easiest.iterrows():
    print(f"     - {row['relation']:25s}: {row['clip_mm_f1']:.3f}")

print(f"\n4. SPATIAL RELATION CATEGORIES:")
print(f"   Vertical (above, below):    Expected harder than egocentric (left, right)")
print(f"   Topological (inside, on):   Expected hardest for distributional, easier for CLIP")
print(f"   Projective (left, right):   Viewpoint-dependent, may resist visual grounding")

print("\n" + "="*70)